### Завдання 1. Підготовка та очищення датасету
Отримати та завантажити набір даних "Individual Household Electric Power Consumption Dataset". Виконати попередню обробку: очистити дані від пропусків. Кожну подальшу вибірку слід оформити у вигляді окремої функції, а для оцінки швидкодії скриптів використати модуль `timeit`.

In [1]:
import pandas as pd
import timeit
import os

# Оскільки ми завантажили файл вручну, просто читаємо його локально
target_file = "household_power_consumption.txt"

if os.path.exists(target_file):
    print("Зчитуємо дані у DataFrame...")
    # Зчитування та очищення від '?'
    main_df = pd.read_csv(target_file, sep=';', na_values=['?'], low_memory=False)
    main_df = main_df.dropna()
    
    print(f"Очищення завершено! Кількість валідних записів: {len(main_df)}")
    display(main_df.head())
else:
    print(f"Файл {target_file} не знайдено! Переконайся, що він лежить поруч із ноутбуком.")

Зчитуємо дані у DataFrame...
Очищення завершено! Кількість валідних записів: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Завдання 2. Фільтрація за потужністю
Сформувати вибірку рядків, де показник загальної активної потужності (`Global_active_power`) становить більше ніж 5 кВт.

In [2]:
def get_high_power_usage(dataframe):
    return dataframe[dataframe['Global_active_power'] > 5.0]

res_task2 = get_high_power_usage(main_df)

exec_time = timeit.timeit(lambda: get_high_power_usage(main_df), number=10) / 10

print(f"Знайдено записів: {len(res_task2)}")
print(f"Середній час виконання: {exec_time:.5f} секунд")
res_task2.head()

Знайдено записів: 17547
Середній час виконання: 0.00476 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


### Завдання 3. Фільтрація за струмом та приладами
Відфільтрувати дані, залишивши ті домогосподарства, де сила струму знаходиться у діапазоні від 19 до 20 А. Серед них виокремити ті записи, де рівень споживання холодильника та пральної машини (`Sub_metering_2`) перевищує витрати на бойлер та кондиціонер (`Sub_metering_3`).

In [3]:
def get_specific_appliances(dataframe):
    condition_current = dataframe['Global_intensity'].between(19.0, 20.0)
    condition_appliances = dataframe['Sub_metering_2'] > dataframe['Sub_metering_3']
    
    return dataframe[condition_current & condition_appliances]

res_task3 = get_specific_appliances(main_df)

exec_time = timeit.timeit(lambda: get_specific_appliances(main_df), number=10) / 10

print(f"Знайдено записів: {len(res_task3)}")
print(f"Середній час виконання: {exec_time:.5f} секунд")
res_task3.head()

Знайдено записів: 2509
Середній час виконання: 0.01210 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


### Завдання 4. Рандомна вибірка та середнє значення
Зробити випадкову вибірку на 500 000 унікальних рядків. На основі цих даних розрахувати середнє значення споживання для кожної з трьох підгруп приладів.

In [4]:
def get_random_sample_means(dataframe):
    random_subset = dataframe.sample(n=500000, replace=False)
    return random_subset[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

means_output = get_random_sample_means(main_df)

exec_time = timeit.timeit(lambda: get_random_sample_means(main_df), number=5) / 5

print("Середні величини груп споживання:")
print(means_output)
print(f"\nСередній час виконання: {exec_time:.5f} секунд")

Середні величини груп споживання:
Sub_metering_1    1.124316
Sub_metering_2    1.301834
Sub_metering_3    6.461452
dtype: float64

Середній час виконання: 0.23264 секунд


### Завдання 5. Багатокритеріальний вечірній фільтр
Знайти всі записи у вечірній час (після 18:00), де споживання перевищує 6 кВт. Далі залишити лише ті випадки, де найбільше енергії витрачається на другу групу приладів. З отриманої вибірки взяти кожен третій рядок з першої половини списку та кожен четвертий — з другої половини.

In [5]:
def evening_peak_filter(dataframe):
    stage1 = dataframe[(dataframe['Time'] >= '18:00:00') & (dataframe['Global_active_power'] > 6.0)]
    
    stage2 = stage1[(stage1['Sub_metering_2'] > stage1['Sub_metering_1']) & 
                    (stage1['Sub_metering_2'] > stage1['Sub_metering_3'])]
    
    midpoint = len(stage2) // 2
    part1 = stage2.iloc[:midpoint]
    part2 = stage2.iloc[midpoint:]
    
    return pd.concat([part1.iloc[::3], part2.iloc[::4]])

res_task5 = evening_peak_filter(main_df)

exec_time = timeit.timeit(lambda: evening_peak_filter(main_df), number=10) / 10

print(f"Знайдено записів після фільтрації та зрізів: {len(res_task5)}")
print(f"Середній час виконання: {exec_time:.5f} секунд")
res_task5.head()

Знайдено записів після фільтрації та зрізів: 310
Середній час виконання: 0.11250 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


### Завдання 6. Статистика та кодування категорій
Виконати нормалізацію та стандартизацію одного з числових стовпців. Розрахувати показники кореляції (за Пірсоном та Спірменом) між двома кількісними параметрами. Застосувати метод One Hot Encoding для перетворення категоріальної змінної.

In [10]:
target_col = 'Global_active_power'
series_min = main_df[target_col].min()
series_max = main_df[target_col].max()

norm_series = (main_df[target_col] - series_min) / (series_max - series_min)
std_series = (main_df[target_col] - main_df[target_col].mean()) / main_df[target_col].std()

print("--- Нормування та Стандартизація ---")
print(f"Оригінал (перші 3):\n{main_df[target_col].head(3).values}")
print(f"Нормовано:\n{norm_series.head(3).values}")
print(f"Стандартизовано:\n{std_series.head(3).values}\n")

pearson_val = main_df['Global_active_power'].corr(main_df['Global_intensity'], method='pearson')
spearman_val = main_df['Global_active_power'].corr(main_df['Global_intensity'], method='spearman')

print("--- Кореляція між Потужністю та Силою струму ---")
print(f"Пірсон: {pearson_val:.4f}")
print(f"Спірмен: {spearman_val:.4f}\n")

print("--- One Hot Encoding ---")
print("Створюємо ознаку 'Month' та кодуємо її...")

main_df['Month'] = main_df['Date'].str.split('/').str[1]
df_encoded = pd.get_dummies(main_df, columns=['Month'])

print(f"Було колонок: {main_df.shape[1]}")
print(f"Стало колонок після OHE: {df_encoded.shape[1]}")

ohe_columns = [c for c in df_encoded.columns if 'Month_' in c]
df_encoded[ohe_columns].head()

--- Нормування та Стандартизація ---
Оригінал (перші 3):
[4.216 5.36  5.374]
Нормовано:
[0.37479631 0.47836321 0.47963064]
Стандартизовано:
[2.95507634 4.03708364 4.05032499]

--- Кореляція між Потужністю та Силою струму ---
Пірсон: 0.9989
Спірмен: 0.9954

--- One Hot Encoding ---
Створюємо ознаку 'Month' та кодуємо її...
Було колонок: 10
Стало колонок після OHE: 21


,Month_1,Month_10,Month_11,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
0,False,False,False,True,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False,False,False,False,False
2,False,False,False,True,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,False,False
4,False,False,False,True,False,False,False,False,False,False,False,False
